databricks free https://dbc-0b69ef93-5247.cloud.databricks.com/editor/notebooks/3801390934062892?o=7474657872577658#command/8499426716608620

In [0]:
# Databricks notebook source
# DBTITLE 1,1. Criando um DataFrame de Exemplo com Valores Nulos
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

# Dados fictícios simulando lacunas em dados reais
data = [
    (1, "Alice", "TI", 8500.0),
    (2, "Bob", None, 6000.0),
    (3, "Charlie", "Vendas", None),
    (4, None, "Marketing", None),
    (5, "Eve", "TI", 9000.0)
]

schema = ["id", "nome", "departamento", "salario"]

df = spark.createDataFrame(data, schema)

print("--- DataFrame Original ---")
df.show()


--- DataFrame Original ---
+---+-------+------------+-------+
| id|   nome|departamento|salario|
+---+-------+------------+-------+
|  1|  Alice|          TI| 8500.0|
|  2|    Bob|        NULL| 6000.0|
|  3|Charlie|      Vendas|   NULL|
|  4|   NULL|   Marketing|   NULL|
|  5|    Eve|          TI| 9000.0|
+---+-------+------------+-------+



In [0]:

# COMMAND ----------

# DBTITLE 2,2. Verificando e Contando Valores Nulos (isNull / isNotNull / count)

print("--- Filtrando linhas onde 'departamento' é nulo ---")
df.filter(col("departamento").isNull()).show()

print("--- Filtrando linhas onde 'salario' NÃO é nulo ---")
df.filter(col("salario").isNotNull()).show()

print("--- Contando valores NÃO nulos por coluna ---")
# Nota: count(col) ignora automaticamente os valores null
df.select(
    count("nome").alias("qtd_nomes_validados"),
    count("departamento").alias("qtd_deptos_validados"),
    count("salario").alias("qtd_salarios_validados")
).show()

--- Filtrando linhas onde 'departamento' é nulo ---
+---+----+------------+-------+
| id|nome|departamento|salario|
+---+----+------------+-------+
|  2| Bob|        NULL| 6000.0|
+---+----+------------+-------+

--- Filtrando linhas onde 'salario' NÃO é nulo ---
+---+-----+------------+-------+
| id| nome|departamento|salario|
+---+-----+------------+-------+
|  1|Alice|          TI| 8500.0|
|  2|  Bob|        NULL| 6000.0|
|  5|  Eve|          TI| 9000.0|
+---+-----+------------+-------+

--- Contando valores NÃO nulos por coluna ---
+-------------------+--------------------+----------------------+
|qtd_nomes_validados|qtd_deptos_validados|qtd_salarios_validados|
+-------------------+--------------------+----------------------+
|                  4|                   4|                     3|
+-------------------+--------------------+----------------------+



In [0]:

# COMMAND ----------

# DBTITLE 3,3. Removendo Linhas com Valores Nulos (dropna / na.drop)

print("--- Removendo linhas com nulos apenas na coluna 'salario' ---")
# Opção 1: df.dropna()
df_sem_nulos_salario = df.dropna(subset=["salario"])
df_sem_nulos_salario.show()

print("--- Removendo linhas com nulos usando a sintaxe alternativa (df.na.drop) ---")
# Opção 2: df.na.drop()
df.na.drop(subset=["departamento", "salario"]).show()

--- Removendo linhas com nulos apenas na coluna 'salario' ---
+---+-----+------------+-------+
| id| nome|departamento|salario|
+---+-----+------------+-------+
|  1|Alice|          TI| 8500.0|
|  2|  Bob|        NULL| 6000.0|
|  5|  Eve|          TI| 9000.0|
+---+-----+------------+-------+

--- Removendo linhas com nulos usando a sintaxe alternativa (df.na.drop) ---
+---+-----+------------+-------+
| id| nome|departamento|salario|
+---+-----+------------+-------+
|  1|Alice|          TI| 8500.0|
|  5|  Eve|          TI| 9000.0|
+---+-----+------------+-------+



In [0]:
# COMMAND ----------

# DBTITLE 4,4. Preenchendo Valores Nulos (fillna / na.fill)

print("--- Preenchendo nulos na coluna 'departamento' com 'Não Informado' ---")
df_preenchido_dept = df.fillna("Não Informado", subset=["departamento"])
df_preenchido_dept.show()

print("--- Preenchendo múltiplos tipos de dados usando um Dicionário (fillna / na.fill) ---")
# Substitui nulos numéricos por 0.0 e textuais por 'Desconhecido'
valores_padrao = {
    "nome": "Anônimo",
    "departamento": "Geral",
    "salario": 0.0
}

df_limpo = df.na.fill(valores_padrao)
df_limpo.show()

--- Preenchendo nulos na coluna 'departamento' com 'Não Informado' ---
+---+-------+-------------+-------+
| id|   nome| departamento|salario|
+---+-------+-------------+-------+
|  1|  Alice|           TI| 8500.0|
|  2|    Bob|Não Informado| 6000.0|
|  3|Charlie|       Vendas|   NULL|
|  4|   NULL|    Marketing|   NULL|
|  5|    Eve|           TI| 9000.0|
+---+-------+-------------+-------+

--- Preenchendo múltiplos tipos de dados usando um Dicionário (fillna / na.fill) ---
+---+-------+------------+-------+
| id|   nome|departamento|salario|
+---+-------+------------+-------+
|  1|  Alice|          TI| 8500.0|
|  2|    Bob|       Geral| 6000.0|
|  3|Charlie|      Vendas|    0.0|
|  4|Anônimo|   Marketing|    0.0|
|  5|    Eve|          TI| 9000.0|
+---+-------+------------+-------+




Type Conversion Under ANSI Mode

Spark 4 enables ANSI SQL mode by default, which changes how invalid conversions behave:

ExpressionSpark 3 (ANSI off)Spark 4 (ANSI on)__CAST('NA' AS INT)__Returns NULL silentlyRaises an error__try_cast('NA' AS INT)__Returns NULLReturns NULLInteger overflowWraps around silentlyRaises an error


⚠️

Use try_cast() for Dirty Data

When cleaning data of unknown quality, use try_cast() (or the Column method col("c").try_cast("int")) so invalid values become NULL that you can then detect and handle, instead of failing the whole job.



## User Defined Functions

UDFs allow you to apply your own Python functions to DataFrame columns:
``
from pyspark.sql.functions import udf
``

### define your function
```
@udf("string")
def make_greeting(name):
    return f"Hello, {name}!"
```

### use your function
```
df.select(make_greeting("name"))
    • UDFs allow developers to create reusable custom functions
    • UDFs can adversely impact performance: they cannot be optimized by the Catalyst optimizer and add serialization overhead between the JVM and the Python worker
    • Always use a built-in function if one is available

```

Pandas UDFs and Apache Arrow

Pandas UDFs let you write Python functions that operate on batches of rows instead of single rows, leveraging Apache Arrow for more efficient Python-JVM serialization.
``
from pyspark.sql.functions import pandas_udf
``
### define your pandas udf
```
@pandas_udf("integer")
def age_plus_one(age_series):
    return age_series + 1
```
### use your pandas udf
```
df.select(age_plus_one("age"))
    • More efficient than their non-pandas counterparts
    • In Spark 4 you can also accelerate row-based UDFs with Arrow serialization: @udf("string", useArrow=True)
    • You should still use a built-in function if one is available - built-ins are always fastest because the optimizer understands them


```





In [0]:
# Databricks notebook source
# DBTITLE 1,1. Configuração Inicial e Dados para Teste de Conversão
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, udf, pandas_udf
import pandas as pd

# DataFrame com dados sujos e inconsistentes para testar a conversão de tipo
dirty_data = [
    ("1", "30", "2147483647"),
    ("2", "45", "100"),
    ("3", "NA", "500"),       # 'NA' causa falha no CAST sob modo ANSI
    ("4", "INVALID", "200")  # Valor textual inválido para inteiro
]

schema = ["id", "idade_str", "valor_grande_str"]
df_dirty = spark.createDataFrame(dirty_data, schema)

print("--- DataFrame Inicial com Dados Sujos ---")
df_dirty.show()


--- DataFrame Inicial com Dados Sujos ---
+---+---------+----------------+
| id|idade_str|valor_grande_str|
+---+---------+----------------+
|  1|       30|      2147483647|
|  2|       45|             100|
|  3|       NA|             500|
|  4|  INVALID|             200|
+---+---------+----------------+



In [0]:
# COMMAND ----------

# DBTITLE 2,2. Conversão de Tipos sob Modo ANSI (cast vs try_cast)

# ⚠️ No Spark 4, o Modo ANSI SQL está ativado por padrão.
# O CAST direto de um valor inválido lançará uma exceção e interromperá o job.
# Exemplo que falharia no Spark 4 / Modo ANSI:
# df_dirty.select(col("idade_str").cast("int")).show()

print("--- Uso seguro de try_cast() para evitar que o job falhe ---")
# O try_cast converte valores inválidos em NULL em vez de estourar erro
df_converted = df_dirty.select(
    col("id"),
    col("idade_str"),
    # Método de coluna
    col("idade_str").try_cast("int").alias("idade_int_col_method"),
    # Sintaxe SQL via expr()
    expr("try_cast(idade_str AS INTEGER)").alias("idade_int_func"),
    # Tratamento de Integer Overflow / Limite de inteiros
    expr("try_cast(valor_grande_str AS INTEGER)").alias("valor_int")
)

df_converted.show()

--- Uso seguro de try_cast() para evitar que o job falhe ---
+---+---------+--------------------+--------------+----------+
| id|idade_str|idade_int_col_method|idade_int_func| valor_int|
+---+---------+--------------------+--------------+----------+
|  1|       30|                  30|            30|2147483647|
|  2|       45|                  45|            45|       100|
|  3|       NA|                NULL|          NULL|       500|
|  4|  INVALID|                NULL|          NULL|       200|
+---+---------+--------------------+--------------+----------+



In [0]:

print("--- Demonstração com SQL Expression ---")
spark.sql("SELECT try_cast('NA' AS INT) as resultado_sql").show()

--- Demonstração com SQL Expression ---
+-------------+
|resultado_sql|
+-------------+
|         NULL|
+-------------+



In [0]:

# COMMAND ----------

# DBTITLE 3,3. Funções Definidas pelo Usuário (UDF Padrão)
# NOTA: UDFs padrão têm sobrecarga de serialização entre a JVM e o Python Worker
# e não recebem otimização do Catalyst Optimizer. Use apenas quando necessário.

@udf("string")
def saudar_usuario(nome: str) -> str:
    if nome is None:
        return "Olá, Visitante!"
    return f"Olá, {nome}!"

df_pessoas = spark.createDataFrame([("Alice",), ("Bob",), (None,)], ["nome"])

print("--- UDF Padrão (Sem otimização Arrow) ---")
df_pessoas.select(col("nome"), saudar_usuario(col("nome")).alias("saudacao")).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:100: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


--- UDF Padrão (Sem otimização Arrow) ---
+-----+---------------+
| nome|       saudacao|
+-----+---------------+
|Alice|    Olá, Alice!|
|  Bob|      Olá, Bob!|
| NULL|Olá, Visitante!|
+-----+---------------+



In [0]:

# COMMAND ----------

# DBTITLE 4,4. Pandas UDFs (Processamento em Lote com Apache Arrow)
# Pandas UDFs operam em séries (batches de linhas) usando Apache Arrow,
# reduzindo drasticamente a sobrecarga de serialização.

@pandas_udf("integer")
def idade_mais_um(idade_series: pd.Series) -> pd.Series:
    # A série do Pandas recebe um lote de dados de uma vez
    return idade_series + 1

df_idades = spark.createDataFrame([(20,), (30,), (40,)], ["idade"])

print("--- Pandas UDF (Vetorizada via Apache Arrow) ---")
df_idades.select(col("idade"), idade_mais_um(col("idade")).alias("idade_proximo_ano")).show()


--- Pandas UDF (Vetorizada via Apache Arrow) ---
+-----+-----------------+
|idade|idade_proximo_ano|
+-----+-----------------+
|   20|               21|
|   30|               31|
|   40|               41|
+-----+-----------------+



In [0]:

# COMMAND ----------

# DBTITLE 5,5. UDF PySpark 4 Acelerada por Arrow vs Função Nativa (Built-in)

# No Spark 4, UDFs linha por linha podem ser aceleradas definindo useArrow=True
@udf("string", useArrow=True)
def formatar_texto_arrow(texto: str) -> str:
    if texto is None:
        return ""
    return texto.strip().upper()

# Exemplo de comparação
df_texto = spark.createDataFrame([("  spark 4  ",), ("  databricks  ",)], ["texto"])

print("--- UDF Acelerada por Arrow (Spark 4) ---")
df_texto.select(col("texto"), formatar_texto_arrow(col("texto")).alias("udf_arrow")).show()


--- UDF Acelerada por Arrow (Spark 4) ---
+--------------+----------+
|         texto| udf_arrow|
+--------------+----------+
|     spark 4  |   SPARK 4|
|  databricks  |DATABRICKS|
+--------------+----------+



In [0]:

print("--- MELHOR PRÁTICA: Usando Funções Nativas (Built-in) ---")
# Sempre prefira funções nativas do PySpark quando existirem.
# Elas rodam diretamente na JVM com velocidade máxima e sem passar pelo Python!
from pyspark.sql.functions import trim, upper

df_texto.select(
    col("texto"),
    upper(trim(col("texto"))).alias("nativo_pyspark")
).show()

--- MELHOR PRÁTICA: Usando Funções Nativas (Built-in) ---
+--------------+--------------+
|         texto|nativo_pyspark|
+--------------+--------------+
|     spark 4  |       SPARK 4|
|  databricks  |    DATABRICKS|
+--------------+--------------+

